# 분류 c

---

## 유형

**투포인터 / 슬라이딩 윈도우**

* 문자열에서 **연속된 부분문자열**을 다룸
* 조건은 **중복 없이 유지**
* 오른쪽 포인터로 확장하다가 조건이 깨지면 왼쪽 포인터로 복구하는 전형적인 슬라이딩 윈도우 문제

---

## 막힌 이유

### 1. 상태 정의를 잘못 잡음

너는 상태를 대충 이렇게 잡았어:

* `log`: 지금까지 나온 문자 기록
* 중복 나오면 `log` 전체 초기화

근데 이 문제의 핵심 상태는
**“현재 윈도우 [left:right]가 중복 없는 상태인지”** 야.

즉 기록 자체보다 중요한 건:

* 현재 윈도우의 왼쪽 경계 `left`
* 현재 문자가 윈도우 안에 있는지
* 조건이 깨졌을 때 왼쪽을 어디까지 옮겨야 하는지

너는 이걸 **“중복 나오면 새로 시작”** 으로 해석했는데,
실제론 **“중복이 사라질 때까지만 왼쪽을 줄이기”** 가 맞음.

---

### 2. 부분문자열을 “통째로 버려야 한다”고 착각함

`"dvdf"`에서 두 번째 `d`를 봤을 때
너는 `"dv"` 전체를 버리고 다시 `"d"`부터 시작했어.

그런데 실제로는:

* 앞의 `d`만 윈도우에서 제거하면
* `"v"`는 계속 살릴 수 있음
* 그래서 `"vdf"`를 만들 수 있음

즉,
**유효한 정보 일부를 보존해야 하는 문제인데 전체 초기화해버린 것**이 핵심 오류야.

---

### 3. 구현 실수라기보다 알고리즘 전환 실패

이건 단순 문법 실수나 조건문 실수보다는
**문제에 맞는 상태 관리 방식으로 전환하지 못한 것**에 가까움.

네 코드는 “기록”은 했지만,
슬라이딩 윈도우의 본질인

* 확장
* 조건 깨짐
* 최소한으로 축소
* 다시 확장

이 흐름으로 안 갔음.

---

## 트리거

다음 신호가 보이면 **투포인터 / 슬라이딩 윈도우**를 떠올리면 된다.

### 신호 1. “부분문자열 / 부분배열”

* 연속된 구간을 다룬다
* 중간을 건너뛰면 안 된다

### 신호 2. “가장 긴 / 가장 짧은”

* 모든 구간을 브루트포스로 보면 비효율적일 가능성이 큼
* 구간을 유지하면서 답을 갱신하는 방식이 필요

### 신호 3. “현재 구간이 조건을 만족해야 함”

이 문제에선 조건이:

* 중복 문자가 없어야 함

즉,
**오른쪽으로 늘리다가 조건 깨지면 왼쪽을 줄여서 복구**
이 패턴이 바로 슬라이딩 윈도우 신호야.

### 이 문제 전용 트리거 한 줄

> “연속된 문자열에서 중복 없이 가장 긴 길이”
> → **윈도우 내부 조건 유지형 슬라이딩 윈도우**

---

## 파이썬 포인트

### 1. `set`

가장 핵심

* 현재 윈도우 안에 문자가 있는지 빠르게 확인
* `in`, `add`, `remove` 사용

예:

```python
seen = set()
```

이 문제에서 `set`은
**현재 윈도우에 어떤 문자가 들어있는지** 관리하는 데 적합함.

---

### 2. `while`

이 문제에서 매우 중요

```python
while s[right] in seen:
    seen.remove(s[left])
    left += 1
```

중복이 사라질 때까지 왼쪽을 줄이는 동작이 필요하므로
`if`가 아니라 `while`이어야 함.

---

### 3. `enumerate`

문자열을 인덱스와 함께 순회할 때 편함

```python
for right, char in enumerate(s):
```

특히 dict로 마지막 위치를 저장하는 풀이에서 자주 씀.

---

### 4. `dict`

심화 풀이에서 사용

* 문자별 마지막 인덱스 저장
* `left`를 한 칸씩 이동하지 않고 점프 가능

예:

```python
last_index = {}
```

---

## 오답노트용으로 짧게 쓰면

### 유형

투포인터 / 슬라이딩 윈도우

### 막힌 이유

중복 발생 시 윈도우를 일부 축소해야 하는데, 전체 초기화한다고 생각해서 상태 정의를 잘못함.
현재 윈도우의 왼쪽 경계 `left`를 관리하지 못함.

### 트리거

연속된 부분문자열 + 가장 긴 길이 + 현재 구간이 특정 조건(중복 없음)을 만족해야 함
→ 슬라이딩 윈도우

### 파이썬 포인트

`set`으로 현재 윈도우 문자 관리,
`while`로 중복이 사라질 때까지 축소,
`enumerate`로 인덱스 순회,
심화는 `dict`로 마지막 위치 저장

---



문자열 s가 주어짐

조합가능한 부분배열중 가장 긴 부분배열의 길이를 출력, 단 부분배열내에 같은 원소가 두개이상 존재해선 안됨(중복 x)

중간에 끊는게 안되는 부분배열이면 sliding window를 만들어 탐색

왼쪽에서 오른쪽으로 확장
새로운 문자를 만날때만 더함
윈도우 안에 이미 존재하는 문자를 만나면 윈도우 초기화

```python
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        
        for 오른쪽으로 한칸씩 문자열을 기록하며 이동:
            매번 어떤 문자인지 : 몇번 나왔는지 기록
            중복이 없으면
                현재 기록의 길이와 현시점 가장 긴 길이(==ans)를 비교해서 더큰 값을 선택 = ans
            두번 나오면
                기록 초기화 하고 현재 문자만 기록 ex) {a:1,b:1,c:1} 에서 a를 만남 => {a:1}

"abcabcbb"

{a:1} ans 1
{a:1,b:1} ans=2
{a:1,b:1,c:1} ans=3
a가 이미 있음 => {a:1} ans = 3
{a:1,b:1} ans = 3
{a:1,b:1,c:1} ans=3
b가 이미 있음 => {b:1}
```

```python
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        log = {}
        ans = 0

        for i in s:
            if i in log:
                log={}
                log[i] = 1
            else:
                log[i] = 1
                ans = max(ans, len(log))
        return ans
        
```

"dvdf" 케이스에서 vdf를 확인못하고 넘어가는 문제 있음


abaa

```python
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        log = {}
        ans = 0
        current_s = ""
        for i in s:
            current_s += i
            if i in log:
                if i == current_s[-2]:
                    current_s = i 
                    log={}
                    log[i] = 1
                else:
                    current_s = i
                    log[i] = {}                     
            else:
                log[i] = 1
                ans = max(ans, len(log))
        return ans
        



```

In [6]:
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        last_index = {}
        left = 0
        ans = 0

        for right, char in enumerate(s):
            if char in last_index and last_index[char] >= left:
                left = last_index[char] + 1

            last_index[char] = right
            ans = max(ans, right - left + 1)

        return ans
    
s = "pwkwed"

test = Solution()

print(test.lengthOfLongestSubstring(s))

4
